# First-Order System Response

**Learning Goals**
- Derive the transfer function of first-order systems (RC circuit, mass-damper)
- Compute and visualize the response to unit-impulse, unit-step, and unit-ramp inputs
- Understand how the time constant $\tau$ governs the system dynamics

## Relevant lecture video

In [1]:
from IPython.display import HTML

HTML('<iframe width="560" height="315" src="https://echo360.org/media/cb199252-3ee5-4cbf-b3e0-3b144cffe9ea/public?autoplay=false&automute=false&currentMediaId=f74ed398-c146-4832-afd7-ce2113f1cd21" frameborder="0" allowfullscreen></iframe>')

/home/matvei/JupyterBasedControlEngineeringTextbook/venv/lib/python3.12/site-packages/IPython/core/display.py:447: UserWarning: Consider using IPython.display.IFrame instead
  warnings.warn("Consider using IPython.display.IFrame instead")


In [2]:
%pip install -q ipywidgets

Note: you may need to restart the kernel to use updated packages.


In [2]:
import ipywidgets as widgets
from IPython.display import display, clear_output
import numpy as np
import matplotlib.pyplot as plt

print("Libraries loaded.")

Libraries loaded.


---
## First-Order Systems

A first-order system has the standard form:

$$H(s) = \frac{K}{\tau s + 1}$$

where $K$ is the **DC gain** and $\tau$ is the **time constant**.

**Responses to standard inputs** ($t \ge 0$):

| Input $u(t)$ | Output $y(t)$ |
|-------------|---------------|
| Unit impulse $\delta(t)$ | $\displaystyle y(t) = \frac{K}{\tau} e^{-t/\tau}$ |
| Unit step $1(t)$ | $\displaystyle y(t) = K \bigl(1 - e^{-t/\tau}\bigr)$ |
| Unit ramp $t$ | $\displaystyle y(t) = K \bigl(t - \tau + \tau e^{-t/\tau}\bigr)$ |

At $t = \tau$, the step response reaches $63.2\%$ of its final value.
At $t = 3\tau$, it reaches $95\%$; at $t = 5\tau$, $99.3\%$.

---
### RC Circuit

For the voltage across the capacitor in a series RC circuit:

$$H(s) = \frac{V_C(s)}{V_{\text{in}}(s)} = \frac{1}{RC\,s + 1}$$

where $\tau = RC$ and $K = 1$.

**Responses:**
- Impulse: $v_C(t) = \dfrac{1}{\tau} e^{-t/\tau}$
- Step: $v_C(t) = 1 - e^{-t/\tau}$
- Ramp: $v_C(t) = t - \tau + \tau e^{-t/\tau}$

In [3]:
R_rc = widgets.FloatSlider(min=100, max=10000, step=100, value=1000,
                           description='R (\u03A9):',
                           style={'description_width': 'initial'})
C_rc = widgets.FloatSlider(min=0.1e-6, max=10e-6, step=0.1e-6, value=1e-6,
                           description='C (F):',
                           readout_format='.2e',
                           style={'description_width': 'initial'})
run_rc = widgets.Button(description='Run', button_style='primary')
out_rc = widgets.Output()

def plot_rc(R, C):
    tau = R * C
    t = np.linspace(0, 5 * tau, 500)

    y_imp = (1 / tau) * np.exp(-t / tau)
    y_step = 1 - np.exp(-t / tau)
    y_ramp = t - tau + tau * np.exp(-t / tau)

    fig, (ax1, ax2, ax3) = plt.subplots(3, 1, figsize=(7, 7.5))

    u_imp = np.zeros_like(t)
    u_imp[0] = 1 / (t[1] - t[0])
    ax1.plot(t, u_imp, 'k--', linewidth=1, alpha=0.5, label='Input $\\delta(t)$')
    ax1.plot(t, y_imp, 'b-', linewidth=2, label='Response')
    ax1.axvline(tau, color='r', linestyle=':', alpha=0.5,
                label=r'$\tau = {:.2e}$ s'.format(tau))
    ax1.set_ylabel('$v_C$ (V)')
    ax1.set_title(r'RC: Unit Impulse Response  $v_C(t) = e^{-t/\tau}/\tau$')
    ax1.grid(True, alpha=0.3)
    ax1.legend(fontsize=8)

    ax2.plot(t, np.ones_like(t), 'k--', linewidth=1, alpha=0.5, label='Input $1(t)$')
    ax2.plot(t, y_step, 'b-', linewidth=2, label='Response')
    ax2.axvline(tau, color='r', linestyle=':', alpha=0.5,
                label=r'$\tau = {:.2e}$ s'.format(tau))
    ax2.axhline(0.632, color='gray', linestyle='--', alpha=0.3,
                label='63.2%')
    ax2.set_ylabel('$v_C$ (V)')
    ax2.set_title(r'RC: Unit Step Response  $v_C(t) = 1 - e^{-t/\tau}$')
    ax2.grid(True, alpha=0.3)
    ax2.legend(fontsize=8)

    ax3.plot(t, t, 'k--', linewidth=1, alpha=0.5, label='Input $t$')
    ax3.plot(t, y_ramp, 'b-', linewidth=2, label='Response')
    ax3.axvline(tau, color='r', linestyle=':', alpha=0.5,
                label=r'$\tau = {:.2e}$ s'.format(tau))
    ax3.set_xlabel('Time (s)')
    ax3.set_ylabel('$v_C$ (V)')
    ax3.set_title(r'RC: Unit Ramp Response  $v_C(t) = t - \tau + \tau e^{-t/\tau}$')
    ax3.grid(True, alpha=0.3)
    ax3.legend(fontsize=8)

    plt.tight_layout()
    plt.show()

def on_run_rc(b):
    with out_rc:
        clear_output(wait=True)
        plot_rc(R_rc.value, C_rc.value)

run_rc.on_click(on_run_rc)
display(widgets.VBox([R_rc, C_rc, run_rc, out_rc]))
print("Adjust R and C, then click Run.")


Adjust R and C, then click Run.


---
### Mass-Damper System

A mass $m$ with damper $b$, driven by force $f(t)$. Output is velocity $v(t)$:

$$f(t) = m\,\dot{v}(t) + b\,v(t)$$

Taking the Laplace transform:

$$F(s) = m\,s\,V(s) + b\,V(s) = V(s)(ms + b)$$

$$H(s) = \frac{V(s)}{F(s)} = \frac{1}{ms + b} = \frac{1/b}{(m/b)s + 1}$$

where $\tau = m/b$ and $K = 1/b$.

**Responses (velocity $v(t)$):**
- Impulse: $v(t) = \dfrac{1}{m} e^{-t/\tau}$
- Step: $v(t) = \dfrac{1}{b}\bigl(1 - e^{-t/\tau}\bigr)$
- Ramp: $v(t) = \dfrac{1}{b}\bigl(t - \tau + \tau e^{-t/\tau}\bigr)$

In [4]:
m_s = widgets.FloatSlider(min=0.1, max=10, step=0.1, value=1,
                          description='m (kg):',
                          style={'description_width': 'initial'})
b_s = widgets.FloatSlider(min=0.1, max=10, step=0.1, value=1,
                          description='b (N\u00b7s/m):',
                          style={'description_width': 'initial'})
run_md = widgets.Button(description='Run', button_style='primary')
out_md = widgets.Output()

def plot_mass_damper(m, b):
    tau = m / b
    K = 1 / b
    t = np.linspace(0, 5 * tau, 500)

    y_imp = (K / tau) * np.exp(-t / tau)
    y_step = K * (1 - np.exp(-t / tau))
    y_ramp = K * (t - tau + tau * np.exp(-t / tau))

    fig, (ax1, ax2, ax3) = plt.subplots(3, 1, figsize=(7, 7.5))

    u_imp = np.zeros_like(t)
    u_imp[0] = 1 / (t[1] - t[0])
    ax1.plot(t, u_imp, 'k--', linewidth=1, alpha=0.5, label='Input $\\delta(t)$')
    ax1.plot(t, y_imp, 'b-', linewidth=2, label='Response')
    ax1.axvline(tau, color='r', linestyle=':', alpha=0.5,
                label=r'$\tau = {:.3f}$ s'.format(tau))
    ax1.set_ylabel('$v$ (m/s)')
    ax1.set_title(r'Mass-Damper: Unit Impulse  $v(t) = e^{-t/\tau}/m$')
    ax1.grid(True, alpha=0.3)
    ax1.legend(fontsize=8)

    ax2.plot(t, K * np.ones_like(t), 'k--', linewidth=1, alpha=0.5,
             label=r'Input $K\cdot 1(t)$')
    ax2.plot(t, y_step, 'b-', linewidth=2, label='Response')
    ax2.axvline(tau, color='r', linestyle=':', alpha=0.5,
                label=r'$\tau = {:.3f}$ s'.format(tau))
    ax2.axhline(K * 0.632, color='gray', linestyle='--', alpha=0.3,
                label='63.2% of $K$')
    ax2.set_ylabel('$v$ (m/s)')
    ax2.set_title(r'Mass-Damper: Unit Step  $v(t) = (1/b)(1 - e^{-t/\tau})$')
    ax2.grid(True, alpha=0.3)
    ax2.legend(fontsize=8)

    ax3.plot(t, K * t, 'k--', linewidth=1, alpha=0.5, label=r'Input $K\cdot t$')
    ax3.plot(t, y_ramp, 'b-', linewidth=2, label='Response')
    ax3.axvline(tau, color='r', linestyle=':', alpha=0.5,
                label=r'$\tau = {:.3f}$ s'.format(tau))
    ax3.set_xlabel('Time (s)')
    ax3.set_ylabel('$v$ (m/s)')
    ax3.set_title(r'Mass-Damper: Unit Ramp  $v(t) = (1/b)(t - \tau + \tau e^{-t/\tau})$')
    ax3.grid(True, alpha=0.3)
    ax3.legend(fontsize=8)

    plt.tight_layout()
    plt.show()

def on_run_md(b):
    with out_md:
        clear_output(wait=True)
        plot_mass_damper(m_s.value, b_s.value)

run_md.on_click(on_run_md)
display(widgets.VBox([m_s, b_s, run_md, out_md]))
print("Adjust m and b, then click Run.")


Adjust m and b, then click Run.


---
## Summary

| System | Transfer Function | Time Constant $\tau$ | DC Gain $K$ |
|--------|------------------|---------------------|-------------|
| RC circuit | $\dfrac{1}{RC\,s + 1}$ | $RC$ | $1$ |
| Mass-damper (velocity) | $\dfrac{1/b}{(m/b)s + 1}$ | $m/b$ | $1/b$ |

Both are first-order systems with the same response shape. Only the time constant and steady-state gain differ.

**Key takeaways:**
- The time constant $\tau$ determines how fast the system responds
- After $5\tau$, the system is effectively at steady state
- The step response reaches $63.2\%$ of its final value at $t = \tau$
- The ramp response lags the input by $\tau$ in steady state